# V6.5-20 — Train student cho Raspberry Pi (YOLO26n + YOLO26s distill từ teacher v5) — chạy được ở mọi máy có GPU

**Cách dùng (giống nhau trên máy local hay GPU thuê, không cần sửa code):**
1. `git clone -b v6p5-plateau-yolo26n …` (hoặc `git pull`), tạo `.env` ở gốc repo có `CAMBER_API_KEY=...`,
   cài Camber CLI (`curl -sL https://cli.cambercloud.com/install-v2.sh | bash`; Windows: đặt `CAMBER_BIN=` trong `.env`).
2. `pip install ultralytics==8.4.172 ncnn pnnx pandas` + torch bản CUDA.
3. Mở notebook → **Run All** (máy thuê qua SSH: chạy bằng `jupyter nbconvert --execute` trong `tmux`, xem README).

Notebook tự: tải dataset `v6p5_v5` (~6 GB, kiểm tra sha256) + teacher v5 + student v5 từ Camber → chọn batch theo VRAM,
workers theo hệ điều hành, cache theo dung lượng ổ → **distill từng student trong `STUDENTS`** từ teacher v5
(mặc định **n**: fine-tune từ student v5, 30 epoch; **s**: từ COCO `yolo26s.pt`, 60 epoch) → export NCNN cho Pi →
eval (val gốc + val_clean, mAP50 + mAP50-95 từng class, MEAN_5 = 5 class lỗi) → upload kết quả lên Camber.

- **Thử nhanh trước** (~10–15 phút gồm tải dataset): `DRY_RUN = True` ở cell 1 hoặc biến môi trường `DRY_RUN=1`.
- **Bị ngắt / tắt máy / treo**: chạy lại (Run All) → stage nào xong thì bỏ qua, đang dở thì resume.
- Teacher v5 (YOLO26m, MEAN_5 val_clean 0.816) được dùng nguyên; `TRAIN_TEACHER = True` để train lại teacher.
- Chọn model cho Pi 3: so MEAN_5 của `student_n_ncnn` / `student_s_ncnn` ở cuối cell 8 với tốc độ đo trên Pi
  (`artifacts/pi_bench_v5/`, YOLO26s nặng hơn n ~3.5 lần).
- Dataset 7 class: insulator, insulator_broken, insulator_flashover, bird_nest, foreign_object, broken_strand, person
  (chi tiết `data/label_audit/README.md`).

In [ ]:
# 1. CẤU HÌNH — mặc định chạy được luôn; chỉ sửa nếu muốn thử khác
import os, sys, platform, shutil, time, csv, json, logging
from pathlib import Path

RUN_NAME = "v6p5_v5_students"         # thư mục artifacts/<RUN_NAME>, trên Camber: <RUN_NAME>_<ngày>
DATASET_NAME = "v6p5_v5"
STASH_DATASET = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_v5"
STASH_INIT = "stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_v5_20261013"   # teacher/best.pt, student_distill/best.pt
STASH_RESULTS = "stash://bunpmc/projects/yolov11sdi/checkpoints"
STUDENTS = ["n", "s"]                 # student cho Pi được train lần lượt (bỏ "n" nếu chỉ cần s, ...)
STUDENT_CFG = {"n": dict(init="v5", epochs=30, patience=10),     # n: fine-tune từ student v5 (0.779 NCNN)
               "s": dict(init="coco", epochs=60, patience=15)}   # s: từ yolo26s.pt (COCO), lâu hơn
IMGSZ_STUDENT = 640
TRAIN_TEACHER = False                 # False = dùng teacher v5 (0.816); True = fine-tune lại teacher từ v5
IMGSZ_TEACHER, EPOCHS_TEACHER = 640, 25
TRAIN_CLASSES = None                  # None = cả 7 class; ví dụ [0, 1, 2, 3, 4, 5] = bỏ person
UPLOAD_RESULTS = os.environ.get("UPLOAD_RESULTS", "0") == "1"
DRY_RUN = os.environ.get("DRY_RUN", "0") == "1"   # hoặc đặt True để thử nhanh
TARGET_MAP50 = 0.82

# ---- tự động: đường dẫn, phần cứng (không cần sửa) ----
import torch, ultralytics
from ultralytics import YOLO
from ultralytics.utils import LOGGER
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
for line in (REPO / ".env").read_text().splitlines() if (REPO / ".env").is_file() else []:   # CAMBER_API_KEY... (không in ra)
    if "=" in line and not line.lstrip().startswith("#"):
        k, v = line.split("=", 1); os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
DATA_DIR = Path(os.environ.get("V6P5_DATA_DIR", REPO / "datasets"))
OUT = REPO / "artifacts" / (RUN_NAME + ("_dryrun" if DRY_RUN else ""))
RUNS, INIT_DIR = OUT / "runs", OUT / "init"
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand", "person"]
DEFECTS = NAMES[1:6]
WINDOWS = platform.system() == "Windows"
CAMBER = os.environ.get("CAMBER_BIN") or shutil.which("camber") or str(Path.home() / ".camber/bin" / ("camber.exe" if WINDOWS else "camber"))

if ultralytics.__version__ != "8.4.172":
    raise RuntimeError(f"Cần ultralytics==8.4.172 (có distill_model); đang có {ultralytics.__version__}: pip install ultralytics==8.4.172")
if torch.cuda.is_available():
    DEVICE, VRAM = 0, torch.cuda.get_device_properties(0).total_memory / 2**30
elif DRY_RUN:
    DEVICE, VRAM = "cpu", 0.0
else:
    raise RuntimeError("Không thấy GPU CUDA (cài torch bản CUDA). Muốn thử trên CPU: DRY_RUN = True")
# batch theo VRAM (khi distill, teacher YOLO26m chạy kèm student mỗi batch nên cũng chiếm VRAM)
BATCH_TEACHER = 24 if VRAM >= 20 else 12 if VRAM >= 11 else 6 if VRAM >= 7 else 4
BATCH_STUDENT = {"n": 32 if VRAM >= 20 else 16 if VRAM >= 11 else 8,
                 "s": 24 if VRAM >= 20 else 12 if VRAM >= 11 else 6}
WORKERS = min(4 if WINDOWS else 8, os.cpu_count() or 2)
DATA_DIR.mkdir(parents=True, exist_ok=True)
FREE_GB = shutil.disk_usage(DATA_DIR).free / 2**30
CACHE = "disk" if FREE_GB >= 70 and not DRY_RUN else False   # cache .npy cần ~55 GB; không đủ ổ thì đọc ảnh trực tiếp
if DRY_RUN:
    IMGSZ_TEACHER = IMGSZ_STUDENT = 320; EPOCHS_TEACHER = 2; WORKERS = 2
    STUDENT_CFG = {a: {**c, "epochs": 2} for a, c in STUDENT_CFG.items()}
    BATCH_TEACHER = 8; BATCH_STUDENT = {a: 8 for a in BATCH_STUDENT}
COMMON = dict(device=DEVICE, workers=WORKERS, seed=20261013, deterministic=True, optimizer="AdamW", lr0=1e-3, lrf=0.05,
              cos_lr=True, weight_decay=1e-3, cache=CACHE, amp=True, pretrained=True, plots=False, save_period=5,
              classes=TRAIN_CLASSES)
print(f"{platform.system()} | GPU: {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'} ({VRAM:.0f} GB) | ultralytics {ultralytics.__version__}")
print(f"students {STUDENTS} ({ {a: (STUDENT_CFG[a]['init'], STUDENT_CFG[a]['epochs'], BATCH_STUDENT[a]) for a in STUDENTS} } = init, epochs, batch) | "
      f"teacher {'train lại' if TRAIN_TEACHER else 'v5 có sẵn'} | workers {WORKERS} | cache {CACHE} (ổ trống {FREE_GB:.0f} GB) | DRY_RUN {DRY_RUN}")
print("data:", DATA_DIR, "| kết quả:", OUT, "| camber:", CAMBER if Path(CAMBER).exists() or shutil.which(CAMBER) else "KHÔNG THẤY")

Windows | GPU: NVIDIA GeForce RTX 4060 Laptop GPU (8 GB) | ultralytics 8.4.172
students ['n', 's'] ({'n': ('v5', 30, 8), 's': ('coco', 60, 6)} = init, epochs, batch) | teacher v5 có sẵn | workers 4 | cache disk (ổ trống 280 GB) | DRY_RUN False
data: d:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\datasets | kết quả: d:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\artifacts\v6p5_v5_students | camber: KHÔNG THẤY


In [2]:
# 2. TẢI DATASET từ Camber (bỏ qua nếu đã có) + kiểm tra sha256 + sửa đường dẫn trong data.yaml
import hashlib, subprocess, tarfile, random, yaml

def camber_cp(src, dst):
    if not (Path(CAMBER).exists() or shutil.which(CAMBER)):
        raise RuntimeError(f"Không thấy Camber CLI ({CAMBER}). Cài CLI hoặc đặt CAMBER_BIN trong .env; "
                           f"hoặc tự tải {src} về {dst}")
    subprocess.run([CAMBER, "stash", "cp", src, str(dst)], check=True, stdout=subprocess.DEVNULL)

DS = DATA_DIR / DATASET_NAME
if not (DS / "manifest.json").is_file():
    tar = DATA_DIR / f"{DATASET_NAME}.tar"
    for f in (f"{DATASET_NAME}.tar.sha256", f"{DATASET_NAME}.package.json", f"{DATASET_NAME}.tar"):
        if not (DATA_DIR / f).is_file():
            print("tải", f, "..."); camber_cp(f"{STASH_DATASET}/{f}", DATA_DIR / f)
    want = (DATA_DIR / f"{DATASET_NAME}.tar.sha256").read_text().split()[0]
    h = hashlib.sha256()
    with tar.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() != want:
        tar.unlink(); raise RuntimeError("sha256 không khớp (tải lỗi) → đã xoá file tar, chạy lại cell này")
    print("giải nén ..."); tarfile.open(tar).extractall(DATA_DIR)
    tar.unlink()  # giữ ổ trống cho cache
for y, v in [("data.yaml", "images/val"), ("data_val_clean.yaml", "images/val_clean")]:   # path trong tar là của máy build
    cfg = yaml.safe_load((DS / y).read_text()); cfg.update(path=str(DS), train="images/train", val=v)
    (DS / y).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
assert [yaml.safe_load((DS / "data.yaml").read_text())["names"][i] for i in range(7)] == NAMES
pkg = json.loads((DATA_DIR / f"{DATASET_NAME}.package.json").read_text())
print("dataset OK:", DS, "| files:", pkg["files"], "| labels_sha256:", pkg["labels_sha256"][:16])

if DRY_RUN:   # bộ mini 160/80/80 ảnh để thử toàn bộ notebook nhanh
    M = DATA_DIR / f"{DATASET_NAME}_mini"; random.seed(0)
    if not (M / "data.yaml").is_file():
        for s, n in [("train", 160), ("val", 80), ("val_clean", 80)]:
            (M / "images" / s).mkdir(parents=True, exist_ok=True); (M / "labels" / s).mkdir(parents=True, exist_ok=True)
            for st in random.sample(sorted(p.stem for p in (DS / "labels" / s).glob("*.txt")), n):
                shutil.copy(DS / "images" / s / f"{st}.jpg", M / "images" / s); shutil.copy(DS / "labels" / s / f"{st}.txt", M / "labels" / s)
        for y, v in [("data.yaml", "images/val"), ("data_val_clean.yaml", "images/val_clean")]:
            (M / y).write_text(yaml.safe_dump({"path": str(M), "train": "images/train", "val": v, "names": dict(enumerate(NAMES))}, sort_keys=False))
    DS = M
DATA_YAML, VAL_CLEAN_YAML = DS / "data.yaml", DS / "data_val_clean.yaml"
COMMON["data"] = str(DATA_YAML)
print("train trên:", DATA_YAML)

tải v6p5_v5.tar.sha256 ...


RuntimeError: Không thấy Camber CLI (C:\Users\dotru\.camber\bin\camber.exe). Cài CLI hoặc đặt CAMBER_BIN trong .env; hoặc tự tải stash://bunpmc/projects/yolov11sdi/datasets/v6p5_v5/v6p5_v5.tar.sha256 về d:\Data-Centric-UAV-Powerline-Defect-Detection\Uav-Powerline-Defect-Detection\datasets\v6p5_v5.tar.sha256

In [ ]:
# 3. WEIGHTS: teacher v5 (+ student v5 cho các student init="v5") từ Camber; student init="coco" ultralytics tự tải
INIT_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_V5, STUDENT_V5 = INIT_DIR / "teacher_v5.pt", INIT_DIR / "student_n_v5.pt"
need = [(TEACHER_V5, "teacher/best.pt")]
if any(STUDENT_CFG[a]["init"] == "v5" for a in STUDENTS):
    need.append((STUDENT_V5, "student_distill/best.pt"))
for dst, src_ in need:
    if not dst.is_file():
        print("tải", src_); camber_cp(f"{STASH_INIT}/{src_}", dst)
    print(dst.name, "→", YOLO(str(dst)).names)
def student_init(a):
    if STUDENT_CFG[a]["init"] == "v5":
        assert a == "n", "student v5 chỉ có bản n; student khác dùng init='coco'"
        return STUDENT_V5
    return f"yolo26{a}.pt"   # COCO pretrained, head tự đổi sang 7 class khi train

In [ ]:
# 4. HÀM TRAIN: resume / bỏ qua nếu xong / watchdog chống treo / tự thử lại, in 1 dòng mỗi epoch
import gc, threading, _thread

def _epoch_line(trainer):
    m, loss = trainer.metrics, trainer.label_loss_items(trainer.tloss)
    final = trainer.stop and getattr(trainer, "_stop_seen", False)   # ultralytics val lại best.pt sau epoch cuối
    trainer._stop_seen = trainer.stop
    tag = "best.pt (val cuối)" if final else f"ep {trainer.epoch + 1:>3}/{trainer.epochs}"
    print(f"{trainer.args.name} {tag}  box {float(loss['train/box_loss']):.3f} cls {float(loss['train/cls_loss']):.3f}  "
          f"mAP50 {m.get('metrics/mAP50(B)', 0):.4f}  mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.4f}  "
          f"lr {trainer.optimizer.param_groups[0]['lr']:.2e}  {time.strftime('%H:%M')}", flush=True)

HANG_MINUTES, MAX_RETRIES = 15, 3

class Watchdog:
    """Ngắt train (như bấm Interrupt) nếu không có batch nào chạy xong trong HANG_MINUTES để cell tự resume."""
    def __init__(self, minutes):
        self.limit, self.beat, self.armed, self.fired = minutes * 60, time.time(), False, False
        self._stop = threading.Event()
    def heartbeat(self, *_):
        self.beat, self.armed = time.time(), True
    def _run(self):
        while not self._stop.wait(30):
            if self.armed and time.time() - self.beat > self.limit:
                self.fired = True
                print(f"[watchdog] {self.limit // 60} phút không có batch nào chạy xong → ngắt để resume", flush=True)
                _thread.interrupt_main()   # chạy được cả Windows (không dùng SIGINT)
                return
    def __enter__(self):
        threading.Thread(target=self._run, daemon=True).start(); return self
    def __exit__(self, *exc):
        self._stop.set()

def _latest_checkpoint(run):
    """last.pt, nếu hỏng (bị kill lúc đang ghi) thì lùi về epochN.pt mới nhất."""
    for p in [run / "weights/last.pt", *sorted(run.glob("weights/epoch*.pt"), key=lambda p: -int(p.stem[5:]))]:
        if p.is_file():
            try:
                return p, torch.load(p, map_location="cpu", weights_only=False)
            except Exception as e:
                print(f"[{run.name}] checkpoint hỏng, bỏ qua {p.name}: {e}")
    return None, None

def train_stage(name, model, **kw):
    run, best = RUNS / name, RUNS / name / "weights/best.pt"
    for attempt in range(1, MAX_RETRIES + 2):
        ckpt, ck = _latest_checkpoint(run)
        targs = (ck or {}).get("train_args", {}) or {}
        if ck is not None and (ck.get("epoch", -1) == -1 or ck["epoch"] + 1 >= targs.get("epochs", 10**9)) \
                and targs.get("epochs", kw.get("epochs")) == kw.get("epochs"):
            print(f"[{name}] đã xong, bỏ qua: {best}"); return best
        if ck is not None and targs.get("epochs") != kw.get("epochs"):
            print(f"[{name}] run cũ {targs.get('epochs')} epoch ≠ {kw.get('epochs')} → bắt đầu lại"); ck = None
        if ck is not None:
            print(f"[{name}] resume từ {ckpt.name} (đã xong {ck['epoch'] + 1} epoch)")
            yolo, args = YOLO(str(ckpt)), dict(resume=True)
        else:
            if run.is_dir(): shutil.rmtree(run)
            print(f"[{name}] bắt đầu: {model}")
            yolo, args = YOLO(str(model)), dict(project=str(RUNS), name=name, exist_ok=False, **{**COMMON, **kw})
        wd = Watchdog(HANG_MINUTES)
        for ev in ("on_train_batch_end", "on_val_batch_end", "on_fit_epoch_end"):
            yolo.add_callback(ev, wd.heartbeat)
        yolo.add_callback("on_fit_epoch_end", _epoch_line)
        level = LOGGER.level; LOGGER.setLevel(logging.WARNING)   # tắt progress bar → output gọn
        try:
            with wd:
                yolo.train(**args)
            print(f"[{name}] xong: {best}"); return best
        except KeyboardInterrupt:
            if not wd.fired:
                print(f"[{name}] dừng theo yêu cầu. Chạy lại cell này để resume."); raise
            print(f"[{name}] treo → resume (lần {attempt}/{MAX_RETRIES})")
        except Exception as e:
            print(f"[{name}] lỗi {type(e).__name__}: {e} → resume (lần {attempt}/{MAX_RETRIES})")
        finally:
            LOGGER.setLevel(level); del yolo; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        time.sleep(30)
    raise RuntimeError(f"[{name}] thất bại sau {MAX_RETRIES} lần thử lại — xem log ở trên")

In [ ]:
# 5. TEACHER — mặc định dùng teacher v5 có sẵn; TRAIN_TEACHER = True để fine-tune lại (từ v5)
if TRAIN_TEACHER:
    TEACHER_BEST = train_stage("teacher", TEACHER_V5, epochs=EPOCHS_TEACHER, batch=BATCH_TEACHER, imgsz=IMGSZ_TEACHER,
                               patience=10, warmup_epochs=1.0, close_mosaic=8)
else:
    TEACHER_BEST, IMGSZ_TEACHER = TEACHER_V5, 640
    print("dùng teacher v5 (YOLO26m, MEAN_5 val_clean 0.816):", TEACHER_BEST)

In [ ]:
# 6. STUDENTS — distill từ teacher, lần lượt theo STUDENTS (teacher chạy kèm mỗi batch nên chậm hơn train thường ~2 lần)
STUDENT_BEST = {}
for a in STUDENTS:
    cfg, ft = STUDENT_CFG[a], STUDENT_CFG[a]["init"] == "v5"
    STUDENT_BEST[a] = train_stage(f"student_{a}", student_init(a), epochs=cfg["epochs"], batch=BATCH_STUDENT[a],
                                  imgsz=IMGSZ_STUDENT, patience=cfg["patience"], warmup_epochs=1.0 if ft else 3.0,
                                  close_mosaic=8 if ft else 15, distill_model=str(TEACHER_BEST))

In [ ]:
# 7. EXPORT NCNN cho Raspberry Pi (mỗi student)
NCNN_DIR = {}
for a in STUDENTS:
    best = RUNS / f"student_{a}" / "weights/best.pt"
    NCNN_DIR[a] = best.with_name("best_ncnn_model")
    if not NCNN_DIR[a].is_dir():
        YOLO(str(best)).export(format="ncnn", imgsz=IMGSZ_STUDENT, half=False)
    print(f"student_{a}: copy sang Pi →", NCNN_DIR[a])
print("Trên Pi: pip install ultralytics ncnn; YOLO('best_ncnn_model', task='detect').predict('anh.jpg', imgsz=%d)" % IMGSZ_STUDENT)

In [ ]:
# 8. EVAL → eval.csv: mAP50 + mAP50-95 từng class, MEAN_5 = 5 class lỗi. Bản NCNN (CPU, chậm) chỉ eval trên val_clean
import pandas as pd
MODELS = {"teacher": (TEACHER_BEST, IMGSZ_TEACHER)}
for a in STUDENTS:
    MODELS[f"student_{a}"] = (RUNS / f"student_{a}" / "weights/best.pt", IMGSZ_STUDENT)
    MODELS[f"student_{a}_ncnn"] = (RUNS / f"student_{a}" / "weights/best_ncnn_model", IMGSZ_STUDENT)
rows = []
for vname, vyaml in [("val", DATA_YAML), ("val_clean", VAL_CLEAN_YAML)]:
    for tag, (w, sz) in MODELS.items():
        w = Path(w)
        if not w.exists() or (w.is_dir() and vname == "val"): continue
        b = YOLO(str(w), task="detect").val(data=str(vyaml), imgsz=sz, batch=1 if w.is_dir() else 16, plots=False, verbose=False,
                                            device="cpu" if w.is_dir() else DEVICE, classes=TRAIN_CLASSES,
                                            project=str(OUT / "eval"), name=f"{tag}_{vname}", exist_ok=True).box
        for i, (c, a50, a95) in enumerate(zip(b.ap_class_index, b.ap50, b.ap)):
            rows.append({"model": tag, "val": vname, "class": NAMES[int(c)], "P": float(b.p[i]), "R": float(b.r[i]),
                         "AP50": float(a50), "AP50_95": float(a95)})
res = pd.DataFrame(rows)
mean5 = res[res["class"].isin(DEFECTS)].groupby(["model", "val"], as_index=False)[["P", "R", "AP50", "AP50_95"]].mean()
res = pd.concat([res, mean5.assign(**{"class": "MEAN_5"})], ignore_index=True)
res.to_csv(OUT / "eval.csv", index=False)
order = ["MEAN_5", *DEFECTS, "insulator", "person"]
for vname in ["val_clean", "val"]:
    t = res[res.val == vname].assign(v=lambda d: d.AP50.round(3).astype(str) + " / " + d.AP50_95.round(3).astype(str))
    print(f"\n=== {vname}: mAP50 / mAP50-95 ===")
    display(t.pivot_table(index="class", columns="model", values="v", aggfunc="first").reindex(order))
for a in STUDENTS:
    pi = res.query(f"model == 'student_{a}_ncnn' and val == 'val_clean' and `class` == 'MEAN_5'")
    if len(pi):
        v = float(pi.AP50.iloc[0])
        print(f"Pi (NCNN) YOLO26{a} val_clean MEAN_5 mAP50 = {v:.4f} →", "ĐẠT" if v > TARGET_MAP50 else "CHƯA ĐẠT", TARGET_MAP50,
              "| v5 n trước đây: 0.779")

In [ ]:
# 9. UPLOAD KẾT QUẢ lên Camber: <STASH_RESULTS>/<RUN_NAME>_<ngày>/
REMOTE = f"{STASH_RESULTS}/{RUN_NAME}_{time.strftime('%Y%m%d')}" + ("_dryrun" if DRY_RUN else "")
stages = (["teacher"] if TRAIN_TEACHER else []) + [f"student_{a}" for a in STUDENTS]
files = {f"{s}/{n}": RUNS / s / p for s in stages
         for n, p in [("best.pt", "weights/best.pt"), ("last.pt", "weights/last.pt"), ("results.csv", "results.csv"), ("args.yaml", "args.yaml")]}
files["eval.csv"] = OUT / "eval.csv"
if UPLOAD_RESULTS:
    for dst, src_ in files.items():
        if src_.is_file():
            camber_cp(str(src_), f"{REMOTE}/{dst}"); print("uploaded", dst)
    for a in STUDENTS:
        if NCNN_DIR[a].is_dir():
            subprocess.run([CAMBER, "stash", "cp", "-r", str(NCNN_DIR[a]), f"{REMOTE}/student_{a}/best_ncnn_model"], check=True,
                           stdout=subprocess.DEVNULL); print(f"uploaded student_{a}/best_ncnn_model")
    print("Kết quả trên Camber:", REMOTE)

In [ ]:
# 10. XEM TIẾN ĐỘ (khi mở lại notebook / kernel đã restart; không ảnh hưởng train) — chạy riêng cell 1 rồi cell này
import datetime as dt
for stage in ["teacher"] + [f"student_{a}" for a in STUDENTS]:
    f = RUNS / stage / "results.csv"
    if not f.exists(): continue
    r_ = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(f.open())]
    b_ = max(r_, key=lambda r: float(r["metrics/mAP50(B)"]))
    print(f"== {stage}: {len(r_)} epoch | best mAP50 {float(b_['metrics/mAP50(B)']):.4f} (ep {b_['epoch']}) "
          f"| cập nhật {dt.datetime.fromtimestamp(f.stat().st_mtime):%H:%M:%S}")
    for r in r_[-5:]:
        print(f"  ep {r['epoch']:>3}  mAP50 {float(r['metrics/mAP50(B)']):.4f}  mAP50-95 {float(r['metrics/mAP50-95(B)']):.4f}")